# 01 — EDA, Qualidade de Dados e Reestruturação PEDE 2025
**Hipótese Científica:** A reestruturação metodológica da base PEDE (2022–2025) altera de forma estatisticamente significativa a distribuição de variáveis educacionais/psicossociais entre ciclos, exigindo particionamento temporal estrito ($t \rightarrow t+1$), validação *out-of-time* e monitorização de *drift* para evitar *data leakage*, *overfitting* e decisões enviesadas.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import settings
from src.pipeline.data_validation import load_and_validate_pede

df_raw, validation_report = load_and_validate_pede()
print(f"Registros totais: {len(df_raw)} | Colunas: {df_raw.shape[1]}")
print("Anos presentes:", sorted(df_raw["ano_referencia"].dropna().unique().tolist()))
print("Relatório de validação:", validation_report)

## 1. Cobertura Temporal e Continuidade por Ciclo
Avaliação da densidade amostral por ciclo (`ano_referencia`) e variação relativa entre períodos.

In [ ]:
cobertura_ano = (
    df_raw.groupby("ano_referencia", dropna=False)
    .size()
    .reset_index(name="total_registros")
    .assign(
        percentual=lambda d: (d["total_registros"] / len(df_raw) * 100).round(2),
        variacao_pct_ciclo=lambda d: d["total_registros"].pct_change().mul(100).round(2),
    )
)
display(cobertura_ano)

## 2. Auditoria de Valores Ausentes e Risco de Viés (MAR/MNAR)
Quantificação de dados ausentes por variável e por ciclo. Em séries longitudinais educacionais, ausências estruturais ou associadas a mudanças de formulário não são puramente aleatórias (MCAR), podendo enviesar estimativas se imputadas fora do tempo.

In [ ]:
nulos_df = (
    pd.DataFrame({
        "qtd_nulos": df_raw.isna().sum(),
        "pct_nulos": (df_raw.isna().mean() * 100).round(2),
        "dtype": df_raw.dtypes.astype(str),
    })
    .sort_values(["pct_nulos", "qtd_nulos"], ascending=False)
)
display(nulos_df.head(15))

nulos_por_ano = (
    df_raw.groupby("ano_referencia", dropna=False)
    .apply(lambda g: g.isna().mean().mean() * 100, include_groups=False)
    .round(2)
)
display(nulos_por_ano.rename("pct_medio_nulos_por_ano").to_frame())

## 3. Teste Não Paramétrico de Shift entre Ciclos (Kolmogorov-Smirnov)
Comparação formal entre os dois últimos ciclos disponíveis (`ks_2samp`) para variáveis numéricas, avaliando deslocamento de distribuição e magnitude de mudança sem pressupor normalidade.

In [ ]:
num_cols = df_raw.select_dtypes(include="number").columns.tolist()
anos = sorted(df_raw["ano_referencia"].dropna().unique().tolist())
if num_cols and len(anos) >= 2:
    a_prev, a_curr = anos[-2], anos[-1]
    df_prev = df_raw[df_raw["ano_referencia"] == a_prev]
    df_curr = df_raw[df_raw["ano_referencia"] == a_curr]
    ks_rows = []
    for col in num_cols:
        s_prev = df_prev[col].dropna()
        s_curr = df_curr[col].dropna()
        if len(s_prev) >= 10 and len(s_curr) >= 10:
            stat, p_val = stats.ks_2samp(s_prev, s_curr)
            ks_rows.append({
                "feature": col,
                f"mediana_{a_prev}": round(float(s_prev.median()), 3),
                f"mediana_{a_curr}": round(float(s_curr.median()), 3),
                "ks_stat": round(float(stat), 4),
                "p_value": float(f"{p_val:.4e}"),
                "shift_significativo_a05": bool(p_val < 0.05),
            })
    ks_df = pd.DataFrame(ks_rows).sort_values("ks_stat", ascending=False)
    display(ks_df.head(12))
else:
    print("Dados insuficientes para teste KS entre ciclos.")

## 4. Síntese Metodológica (Nível PhD)
* **Validação e Risco de Leakage:** O uso de informações futuras ($t+1$) ou normalizações/imputações calculadas no conjunto completo infla artificialmente métricas e mascara *overfitting* temporal.
* **Risco de Viés e Limitações:** Mudanças de critério ou cobertura entre ciclos podem gerar *domain shift* e alterações na taxa-base de risco; o modelo apoia a triagem psicopedagógica (*Human-in-the-Loop*), sem substituir avaliação clínica/pedagógica.